Chroma
- It is Vector Store
- it is lightweight, open source, vector database for local developers and small to medium scall production needs
- it comes in between vector store and vector database as chroma is lighweight but it is having some of the database like features as well

# `Detailed Notes`

# ChromaDB — Detailed Explanation

**ChromaDB (Chroma)** is an open-source database designed for **AI applications that need to store and search embeddings**.

It is commonly used in:

* RAG applications
* PDF chatbots
* Semantic search
* Document question-answering
* Recommendation systems
* AI agents
* Knowledge bases

The easiest way to understand ChromaDB is:

> **ChromaDB stores vectors along with their documents and metadata, and retrieves the most relevant records when you provide a query.**

---

# 1. Why Do We Need ChromaDB?

Suppose you have a 500-page PDF.

```text
Machine Learning Book
        │
        ▼
     500 pages
        │
        ▼
    Text chunks
        │
        ▼
    Embeddings
        │
        ▼
    ChromaDB
```

Now the user asks:

> "What is gradient descent?"

You don't want to send all 500 pages to the LLM.

Instead:

```text
Question
   ↓
Embedding
   ↓
ChromaDB
   ↓
Find similar chunks
   ↓
Top relevant chunks
   ↓
LLM
   ↓
Answer
```

This is the foundation of **RAG**.

---

# 2. What Does ChromaDB Store?

A Chroma collection can associate several pieces of information with each record:

```text
ID
Document
Embedding
Metadata
```

For example:

```text
ID:
doc_001

Document:
"Gradient descent is an optimization algorithm..."

Embedding:
[0.21, -0.73, 0.45, ...]

Metadata:
{
    "source": "ml.pdf",
    "page": 42
}
```

So conceptually:

```text
┌─────────────────────────────────────────┐
│                ChromaDB                 │
├──────────┬─────────────┬───────────────┤
│ ID       │ Embedding   │ Document      │
├──────────┼─────────────┼───────────────┤
│ doc_001  │ [0.21,...]  │ Gradient...   │
│ doc_002  │ [0.51,...]  │ Neural...     │
│ doc_003  │ [0.32,...]  │ Transformer.. │
└──────────┴─────────────┴───────────────┘
```

Metadata can additionally describe each document.

---

# 3. ChromaDB Is Not the Embedding Model

This is a very important distinction.

ChromaDB **does not magically understand text**.

You need an embedding model to convert text into vectors.

```text
Text
 ↓
Embedding Model
 ↓
Vector
 ↓
ChromaDB
```

For example:

```text
"Machine Learning"
        ↓
Embedding Model
        ↓
[0.23, 0.71, -0.41, ...]
        ↓
ChromaDB
```

---

# 4. ChromaDB vs Embedding Model

| Component       | Job                          |
| --------------- | ---------------------------- |
| Embedding Model | Converts text → vectors      |
| ChromaDB        | Stores and searches vectors  |
| Retriever       | Retrieves relevant documents |
| LLM             | Generates final answer       |

Therefore:

```text
Embedding Model ≠ ChromaDB
```

---

# 5. How ChromaDB Works

Suppose you have:

```text
Document 1:
"Machine Learning is a subset of AI."

Document 2:
"Deep Learning uses neural networks."

Document 3:
"Transformers use self-attention."

Document 4:
"Python is a programming language."
```

### Step 1 — Generate embeddings

```text
Document 1 → Vector 1
Document 2 → Vector 2
Document 3 → Vector 3
Document 4 → Vector 4
```

### Step 2 — Store vectors

```text
                 ChromaDB

Vector 1 → Document 1
Vector 2 → Document 2
Vector 3 → Document 3
Vector 4 → Document 4
```

### Step 3 — User asks

```text
"What is Deep Learning?"
```

The query is also embedded:

```text
Question
   ↓
Embedding Model
   ↓
Query Vector
```

### Step 4 — Similarity Search

Chroma compares the query vector with stored vectors.

Conceptually:

```text
Query
 │
 ├── Document 1 → 0.72
 ├── Document 2 → 0.94  ← Relevant
 ├── Document 3 → 0.67
 └── Document 4 → 0.21
```

### Step 5 — Return relevant documents

```text
Document 2
```

Then the retrieved document can be passed to the LLM.

---

# 6. ChromaDB Architecture in RAG

The complete architecture:

```text
                    INGESTION
                        │
                        ▼
                     PDF
                        │
                        ▼
                Document Loader
                        │
                        ▼
                  Text Splitter
                        │
                        ▼
                  Text Chunks
                        │
                        ▼
                Embedding Model
                        │
                        ▼
                    ChromaDB
                        │
                        │
────────────────────────┼────────────────────
                        │
                     QUERY
                        │
                        ▼
                 User Question
                        │
                        ▼
                Embedding Model
                        │
                        ▼
                  Query Vector
                        │
                        ▼
                    ChromaDB
                        │
                        ▼
                Similarity Search
                        │
                        ▼
                Relevant Chunks
                        │
                        ▼
                      LLM
                        │
                        ▼
                     Answer
```

---

# 7. What is a Collection?

A **collection** is a logical grouping of records inside Chroma.

Think of it somewhat like a table or collection in a conventional database.

Example:

```text
ChromaDB
│
├── company_documents
├── research_papers
├── product_manuals
└── customer_support
```

You might create:

```python
collection = client.get_or_create_collection(
    name="company_documents"
)
```

Then store your documents in that collection.

---

# 8. Basic ChromaDB Installation

The current Python package is commonly installed with:

```bash
pip install chromadb
```

For LangChain integration, you will typically also install the LangChain Chroma integration:

```bash
pip install langchain-chroma
```

And your embedding provider package separately.

---

# 9. Basic ChromaDB Example

You can use Chroma directly without LangChain.

```python
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="documents"
)
```

Now you have a collection.

---

# 10. Adding Documents

You can add documents with IDs.

```python
collection.add(
    ids=["1", "2", "3"],
    documents=[
        "Machine Learning is a subset of AI.",
        "Deep Learning uses neural networks.",
        "Transformers use self-attention."
    ]
)
```

Chroma can generate embeddings through its configured/default embedding function, or you can provide embeddings depending on how you configure the collection.

---

# 11. Querying ChromaDB

Now ask:

```python
results = collection.query(
    query_texts=["What is Deep Learning?"],
    n_results=2
)
```

Conceptually:

```text
Question
   ↓
Embedding
   ↓
ChromaDB
   ↓
Top 2 similar documents
```

---

# 12. Adding Metadata

Metadata is extremely useful.

```python
collection.add(
    ids=["1"],
    documents=[
        "Machine Learning is a subset of AI."
    ],
    metadatas=[
        {
            "source": "ml.pdf",
            "page": 10,
            "category": "machine-learning"
        }
    ]
)
```

Now Chroma knows:

```text
Document
+
Source
+
Page
+
Category
```

---

# 13. Metadata Filtering

Suppose you have:

```text
document 1 → HR
document 2 → Engineering
document 3 → Finance
```

You can query with metadata constraints supported by Chroma.

Conceptually:

```text
Find documents about:
"leave policy"

where:

department = HR
```

This is useful in enterprise RAG systems.

---

# 14. Persistence

A basic in-memory Chroma client is useful for experiments.

For persistent local storage, you can configure a persistent client:

```python
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)
```

Now data can be persisted locally rather than existing only for the lifetime of the process.

Conceptually:

```text
Python Application
       │
       ▼
    ChromaDB
       │
       ▼
 ./chroma_db/
```

This is useful for local RAG projects.

---

# 15. ChromaDB with LangChain

This is especially important for your GenAI learning.

LangChain provides a Chroma integration.

```python
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_store = Chroma(
    collection_name="documents",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)
```

Now LangChain can use Chroma as its vector store backend.

---

# 16. Adding Documents Through LangChain

Suppose you already have:

```python
documents
```

Then:

```python
vector_store.add_documents(documents)
```

Or you can create the store directly:

```python
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
```

Architecture:

```text
LangChain
    │
    ▼
Chroma VectorStore
    │
    ▼
ChromaDB
```

---

# 17. Similarity Search Through LangChain

```python
results = vector_store.similarity_search(
    "What is Deep Learning?",
    k=3
)
```

Then:

```python
for doc in results:
    print(doc.page_content)
```

---

# 18. ChromaDB as a Retriever

You can convert the vector store into a retriever:

```python
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)
```

Then:

```python
docs = retriever.invoke(
    "What is Deep Learning?"
)
```

Now:

```text
User Question
      ↓
Retriever
      ↓
ChromaDB
      ↓
Top 3 Documents
```

---

# 19. ChromaDB + LLM

Now we can build a basic RAG pipeline:

```text
User Question
      │
      ▼
Retriever
      │
      ▼
ChromaDB
      │
      ▼
Relevant Documents
      │
      ▼
Prompt
      │
      ▼
Chat Model
      │
      ▼
Answer
```

The LLM receives:

```text
Context:
[retrieved documents]

Question:
What is Deep Learning?
```

Then generates the answer.

---

# 20. ChromaDB vs Traditional Database

This is an important interview comparison.

### MySQL/PostgreSQL

Good for structured queries:

```text
WHERE age > 25
WHERE department = 'Engineering'
```

### ChromaDB

Good for semantic similarity:

```text
Find documents
whose meaning is similar to:
"What is deep learning?"
```

So:

```text
SQL Database
    ↓
Exact/structured queries

Vector Database
    ↓
Similarity/semantic queries
```

Modern systems often use **both**.

---

# 21. ChromaDB vs FAISS

These are often confused.

| Feature               | ChromaDB                                        | FAISS                              |
| --------------------- | ----------------------------------------------- | ---------------------------------- |
| Primary role          | Vector database/vector store                    | Vector similarity search library   |
| Stores documents      | Yes                                             | Not inherently                     |
| Metadata              | Yes                                             | Not its core purpose               |
| Persistence           | Supported                                       | Requires your own storage strategy |
| Database-like API     | Yes                                             | No                                 |
| Local use             | Excellent                                       | Excellent                          |
| Production scaling    | Limited compared with dedicated distributed DBs | Not a database service             |
| LangChain integration | Yes                                             | Yes                                |

A useful mental model:

```text
FAISS

Vector
 ↓
Similarity Search


ChromaDB

Vector
+
Document
+
Metadata
 ↓
Storage + Search
```

---

# 22. ChromaDB vs Pinecone

Both can be used as vector storage systems, but their typical deployment models differ.

### ChromaDB

Good for:

* Learning
* Prototyping
* Local applications
* Small/medium RAG applications
* Development

### Pinecone

Designed as a managed cloud vector database service.

Useful when you need:

* Managed infrastructure
* Production deployments
* Scaling
* Cloud-based operations

---

# 23. ChromaDB vs Qdrant

Qdrant is another dedicated vector database.

For production GenAI systems, you should understand concepts such as:

```text
Collections
Vectors
Payload/Metadata
Filtering
Similarity Search
Indexes
Persistence
```

Chroma is often easier to start with, while Qdrant is a strong option when you need a more dedicated vector database architecture.

---

# 24. What ChromaDB Does NOT Do

This is important.

ChromaDB does **not**:

❌ Generate answers like GPT.

❌ Understand language by itself.

❌ Replace the LLM.

❌ Replace the embedding model.

❌ Automatically make your RAG application accurate.

Instead:

```text
Embedding Model
      ↓
Creates vectors

ChromaDB
      ↓
Stores/searches vectors

Retriever
      ↓
Selects relevant documents

LLM
      ↓
Generates answer
```

---

# 25. ChromaDB in Your PDF Chatbot

Since you're learning RAG, imagine your project:

```text
pdf_chatbot/
│
├── data/
│   └── book.pdf
│
├── ingestion/
│   ├── loader.py
│   ├── splitter.py
│   └── embeddings.py
│
├── vectorstore/
│   └── chroma_db/
│
├── retrieval/
│   └── retriever.py
│
├── generation/
│   └── llm.py
│
└── app.py
```

Workflow:

```text
PDF
 ↓
PyPDFLoader
 ↓
RecursiveCharacterTextSplitter
 ↓
Embedding Model
 ↓
ChromaDB
 ↓
Retriever
 ↓
LLM
 ↓
Answer
```

---

# 26. Important Interview Questions

### Q1. What is ChromaDB?

> ChromaDB is an open-source vector database/vector store designed for AI applications. It stores embeddings along with documents and metadata and supports similarity-based retrieval.

### Q2. Is ChromaDB an LLM?

> No. ChromaDB stores and retrieves vectors. An LLM generates the final response.

### Q3. Does ChromaDB generate embeddings?

> It can use an embedding function depending on configuration, but the conceptual architecture separates the embedding model from the vector database. In LangChain, you commonly explicitly provide an embedding model to the Chroma vector store.

### Q4. Why use ChromaDB in RAG?

> To store document embeddings and efficiently retrieve chunks semantically similar to a user's query.

### Q5. What is a collection?

> A logical grouping of vector records and associated data inside Chroma.

### Q6. What does metadata do?

> Metadata provides additional information about stored documents and can be used for filtering and source attribution.

---

# 27. The Most Important Architecture to Remember

```text
                  RAG SYSTEM

                    USER
                      │
                      ▼
                 QUESTION
                      │
                      ▼
              Embedding Model
                      │
                      ▼
                Query Vector
                      │
                      ▼
                  ChromaDB
                      │
              Similarity Search
                      │
                      ▼
              Relevant Chunks
                      │
                      ▼
                   Prompt
                      │
                      ▼
                  Chat LLM
                      │
                      ▼
                  RESPONSE
```

And during ingestion:

```text
             DOCUMENT
                 │
                 ▼
           Document Loader
                 │
                 ▼
            Text Splitter
                 │
                 ▼
               Chunks
                 │
                 ▼
          Embedding Model
                 │
                 ▼
              Vectors
                 │
                 ▼
             ChromaDB
```

## One-line takeaway

**ChromaDB is the storage and retrieval layer in a RAG system: the embedding model converts text into vectors, ChromaDB stores and searches those vectors, the retriever selects relevant chunks, and the LLM uses those chunks to generate the final answer.**
